In [0]:
# Silver layer: leads (marketing / B2B sales leads)
# Source: b2b_ecommerce.bronze.leads (append-only raw Excel ingestion,
#        every column arrives as text)
# Grain:  one row per lead_id (each lead exactly once)
# Approach: bronze.leads is all-STRING, so we clean + standardize every row
#           (typed dates, currency-free numerics, canonical source/country
#           categories) and MERGE into b2b_ecommerce.silver.leads with
#           AUTO CDC keyed by lead_id. The 27 lead_ids that appear twice
#           in bronze are identical double-writes (same source_file, same
#           updated_at), not versions - so SCD Type 1 (latest wins) is
#           enough to deduplicate them; no history exists to keep.
#           Rows failing the drop-level data quality rules are NOT lost:
#           they are captured in b2b_ecommerce.quarantine.leads_silver
#           (dedicated quarantine schema, <table>_<layer> naming) with the
#           violated rule names in failed_checks.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_LEADS = "b2b_ecommerce.bronze.leads"


def clean_text(name, lower=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if lower:
        trimmed = F.lower(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


def parse_date(name):
    """Excel dates arrive in two formats: 'yyyy-MM-dd HH:mm:ss' (most
    files) and 'dd/MM/yyyy' (61 rows). try_to_date returns NULL instead of
    throwing on the format that does not match; coalesce picks whichever
    parses (all 1138 rows parse with this pair)."""
    return (
        F.coalesce(
            F.expr(f"try_to_date(trim({name}), 'yyyy-MM-dd HH:mm:ss')"),
            F.expr(f"try_to_date(trim({name}), 'dd/MM/yyyy')"),
        )
        .cast("date")
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned + standardized stream of bronze.leads
# ---------------------------------------------------------------------------
# Drop-level rules: a row failing ANY of these is excluded from
# b2b_ecommerce.silver.leads but captured in leads_quarantine below.
# Single source of truth: the expectations on leads_clean_stream AND the
# quarantine table's filter/reason column are all generated from this list,
# so the two can never drift apart.
DROPPED_CHECKS = [
    ("lead_id_not_null", "lead_id IS NOT NULL"),  # merge key must exist
    ("lead_id_format_valid", "lead_id RLIKE '^LD-[0-9]{8}-[0-9]{3}$'"),
    ("updated_at_not_null", "updated_at IS NOT NULL"),  # CDC sequence must exist
    ("created_date_not_null", "created_date IS NOT NULL"),  # also catches unparseable dates
    ("created_date_not_future", "created_date <= CURRENT_DATE()"),  # 4 future-dated rows in bronze
    ("source_valid", "source IN ('Google Ads', 'LinkedIn', 'Trade Show', 'Referral', 'Facebook', 'Email', 'Organic')"),
    ("status_valid", "status IN ('New', 'Contacted', 'Qualified', 'Proposal', 'Won', 'Lost')"),
    ("country_valid", "country IN ('Argentina', 'Chile', 'Brazil', 'Uruguay')"),
    ("company_size_valid", "company_size IN ('1-10', '11-50', '51-200', '201-500', '500+')"),
    ("estimated_order_volume_valid", "estimated_order_volume IS NULL OR estimated_order_volume >= 0"),
    # Referential integrity: a lead cannot be contacted before it was
    # created (12 rows in bronze). There is deliberately NO FK to
    # silver.customers or silver.companies: leads are pre-customer
    # prospects - only ~18% of the contact emails resolve to customers
    # and ~13% of the company names to companies, so an FK check would
    # reject the whole table.
    ("last_contacted_after_created", "last_contacted_date IS NULL OR last_contacted_date >= created_date"),
]

# Warn-level rules: the row is kept in silver and the problem only
# recorded in the event log (same policy as silver.customers checks).
WARNED_CHECKS = [
    ("contact_email_valid", "contact_email IS NULL OR contact_email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'"),  # 32 rows missing '@' in bronze
    ("phone_format_valid", "phone IS NULL OR phone RLIKE '^\\+?[0-9 ()-]+$'"),
    ("company_name_not_null", "company_name IS NOT NULL"),
    ("contact_name_not_null", "contact_name IS NOT NULL"),
]


def cleaned_leads(bronze):
    """Clean + standardize one bronze.leads row. No expectations here on
    purpose: the same transformation must feed both silver.leads (which
    drops invalid rows) and leads_quarantine (which keeps exactly those
    rows)."""
    return bronze.select(
        # Column types: the merge key is an id, not a number - keep text
        clean_text("lead_id"),
        # Cleaning: two-format Excel dates -> proper DATE
        parse_date("created_date"),
        # Standardization: 28 raw variants (case + padding) -> 7 canonical
        # values; anything else -> NULL and caught by source_valid
        F.when(F.lower(F.trim("source")) == "google ads", F.lit("Google Ads"))
         .when(F.lower(F.trim("source")) == "linkedin", F.lit("LinkedIn"))
         .when(F.lower(F.trim("source")) == "trade show", F.lit("Trade Show"))
         .when(F.lower(F.trim("source")) == "referral", F.lit("Referral"))
         .when(F.lower(F.trim("source")) == "facebook", F.lit("Facebook"))
         .when(F.lower(F.trim("source")) == "email", F.lit("Email"))
         .when(F.lower(F.trim("source")) == "organic", F.lit("Organic"))
         .otherwise(F.lit(None).cast("string"))
         .alias("source"),
        clean_text("campaign_name"),
        clean_text("utm_source", lower=True),
        clean_text("utm_medium", lower=True),
        clean_text("utm_campaign", lower=True),
        # Standardization: 'AR'/'CHILE'/'argentina' variants -> canonical names
        F.when(F.lower(F.trim("country")).isin("ar", "argentina"), F.lit("Argentina"))
         .when(F.lower(F.trim("country")).isin("cl", "chile"), F.lit("Chile"))
         .when(F.lower(F.trim("country")).isin("br", "brazil"), F.lit("Brazil"))
         .when(F.lower(F.trim("country")).isin("uy", "uruguay"), F.lit("Uruguay"))
         .otherwise(F.lit(None).cast("string"))
         .alias("country"),
        clean_text("city"),
        clean_text("company_name"),  # 20 rows had padding in bronze
        clean_text("company_size"),
        clean_text("industry"),
        # lead_score: valid only as 0-100 whole number. 'N/A' (56 rows) and
        # out-of-range values (-5, 150 = 28 rows) are nullified instead of
        # dropping the lead - the score is unknown, the lead is still real.
        F.when(
            F.expr("try_cast(trim(lead_score) as int)").between(0, 100),
            F.expr("try_cast(trim(lead_score) as int)"),
        ).otherwise(F.lit(None).cast("int")).alias("lead_score"),
        # estimated_order_volume: '$1,299' currency strings -> number;
        # 'N/A' (26 rows) -> NULL
        F.expr(
            "try_cast(regexp_replace(trim(estimated_order_volume), '[$,]', '') as decimal(12,2))"
        ).alias("estimated_order_volume"),
        clean_text("status"),
        clean_text("utm_content", lower=True),
        clean_text("contact_name"),
        # canonical lowercase email (same form as silver.customers);
        # invalid emails are kept and only warned about
        clean_text("contact_email", lower=True),
        clean_text("phone"),
        clean_text("job_title"),
        clean_text("sales_rep"),
        # NULL = never contacted yet (296 rows in bronze)
        parse_date("last_contacted_date"),
        F.col("source_file"),
        F.col("updated_at").cast("timestamp").alias("updated_at"),
    )


@dp.temporary_view(name="leads_clean_stream")
# Data quality validation (only checks that can actually catch bad rows).
# Drop-level: invalid rows are excluded from silver so downstream reads
# get a clean table (accuracy/validity enforced at the silver boundary) -
# and they are captured in leads_quarantine below, so nothing vanishes.
@dp.expect_all_or_drop(dict(DROPPED_CHECKS))
# Warn-level: the row is kept and the problem only recorded in the event log.
@dp.expect_all(dict(WARNED_CHECKS))
def leads_clean_stream():
    # bronze.leads is append-only: every change arrives as a new row.
    # The AUTO CDC merge below turns each re-send into an in-place update
    # of the same lead_id (latest updated_at wins), which deduplicates
    # the double-writes and keeps re-ingestion idempotent.
    return cleaned_leads(spark.readStream.table(BRONZE_LEADS))


# ---------------------------------------------------------------------------
# 2) Quarantine: b2b_ecommerce.quarantine.leads_silver
#    Every row excluded from silver.leads lands here instead of vanishing:
#    one append-only row per offending bronze row (double-write copies
#    included), cleaned to the same schema as silver.leads plus a
#    failed_checks column naming the rules it violated. Raw text stays in
#    bronze.leads; this table is the review/worklist.
#    Reads bronze directly, NOT leads_clean_stream: expectations attached
#    to that view would drop the bad rows before they could reach here.
# ---------------------------------------------------------------------------
@dp.table(
    # fully-qualified name: publishes outside the pipeline's silver
    # schema into the dedicated quarantine schema, as <table>_<layer>
    name="b2b_ecommerce.quarantine.leads_silver",
    comment=(
        "Leads rejected by silver.leads data quality rules, kept for "
        "inspection and rework; failed_checks lists the violated rules"
    ),
)
def leads_quarantine():
    # IS TRUE matches expect_or_drop semantics: a NULL predicate (e.g. an
    # unmapped source value) counts as a failure, not as a pass.
    quarantine_filter = "NOT (" + " AND ".join(
        f"({predicate}) IS TRUE" for _, predicate in DROPPED_CHECKS
    ) + ")"
    failed_checks = F.concat_ws(
        ", ",
        *[
            F.when(F.expr(f"NOT (({predicate}) IS TRUE)"), F.lit(name))
            for name, predicate in DROPPED_CHECKS
        ],
    )
    return (
        cleaned_leads(spark.readStream.table(BRONZE_LEADS))
        .filter(quarantine_filter)
        .withColumn("failed_checks", failed_checks)
    )


# ---------------------------------------------------------------------------
# 3) Silver target: b2b_ecommerce.silver.leads
#    Rows are MERGEd by lead_id: double-writes collapse into one row and
#    re-ingestion is idempotent. SCD Type 1: immutable lead records, no
#    version history kept.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="leads",
    comment=(
        "Silver leads: cleaned, standardized and deduplicated; one row per "
        "lead_id (SCD Type 1 merge - latest updated_at wins)"
    ),
)

dp.create_auto_cdc_flow(
    target="leads",
    source="leads_clean_stream",
    keys=["lead_id"],
    sequence_by="updated_at",
    # SCD Type 1 (default): no version history exists in the source - the
    # only duplicates are identical double-writes.
)